In [5]:
import os
from dotenv import load_dotenv
load_dotenv()
# load all the content of .env file into the environment variables

if os.environ['GROQ_API_KEY']:
    print("GROQ_API_KEY is set")
else:
    print("API_KEY is not set")

GROQ_API_KEY is set


In [6]:
# from langchain_groq import 
# from langchain_openai import ChatOpenAI

# for groq
from langchain_groq import ChatGroq

/Users/iwasbinod/Desktop/rag_binod/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:

#  llm=ChatOpenAI(model="gpt-5-nano", temperature=0)
# for groq

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

In [8]:
response=llm.invoke("What is the capital of France?")
print(response.content)

The capital of France is Paris.


# RAG implementation with pdf

#### step 1 : Extracting PDF from our  own pdf's .

In [16]:
from langchain_community.document_loaders import PyPDFLoader
pdf_path="../docs/research.pdf"
loader = PyPDFLoader(pdf_path)
docs=loader.load()
docs
# pages = loader.load_and_split()
# print(pages[0].page_content)


[Document(metadata={'producer': 'macOS Version 26.3.1 (Build 25D2128) Quartz PDFContext', 'creator': 'WPS Writer', 'creationdate': '2026-03-12T11:21:28+05:45', 'author': 'Dell', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2026-03-12T11:21:28+05:45', 'sourcemodified': "D:20260312112128+05'45'", 'subject': '', 'title': '', 'trapped': '/False', 'source': '../docs/research.pdf', 'total_pages': 33, 'page': 0, 'page_label': '1'}, page_content='The Role of AI in Shaping Perceived Academic Performance:\nInsights from Engineering Students in Sudurpaschim\nProvince, Nepal\nSUBMITTED BY\nBinod Raj Pant\nDipak Singh Mahar\nHemant Chaudhary\nShekhar Bhatt\nSUBMITTED TO\nDepartment of Research and HRD\nNational Academy of Science and Technology\nDhangadhi-04, Uttarbehedi, Kailali, Nepal\nBE Computer, 7th Semester\n2025-2026'),
 Document(metadata={'producer': 'macOS Version 26.3.1 (Build 25D2128) Quartz PDFContext', 'creator': 'WPS Writer', 'creationdate': '2026-03-12T11:21:28+05:45', 

#### step 2 : Splitting the Document

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'producer': 'macOS Version 26.3.1 (Build 25D2128) Quartz PDFContext', 'creator': 'WPS Writer', 'creationdate': '2026-03-12T11:21:28+05:45', 'author': 'Dell', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2026-03-12T11:21:28+05:45', 'sourcemodified': "D:20260312112128+05'45'", 'subject': '', 'title': '', 'trapped': '/False', 'source': '../docs/research.pdf', 'total_pages': 33, 'page': 0, 'page_label': '1'}, page_content='The Role of AI in Shaping Perceived Academic Performance:\nInsights from Engineering Students in Sudurpaschim\nProvince, Nepal\nSUBMITTED BY\nBinod Raj Pant\nDipak Singh Mahar\nHemant Chaudhary\nShekhar Bhatt\nSUBMITTED TO\nDepartment of Research and HRD\nNational Academy of Science and Technology\nDhangadhi-04, Uttarbehedi, Kailali, Nepal\nBE Computer, 7th Semester\n2025-2026'),
 Document(metadata={'producer': 'macOS Version 26.3.1 (Build 25D2128) Quartz PDFContext', 'creator': 'WPS Writer', 'creationdate': '2026-03-12T11:21:28+05:45', 

In [19]:
len(chunks) 

82

In [21]:
chunks[0].metadata

{'producer': 'macOS Version 26.3.1 (Build 25D2128) Quartz PDFContext',
 'creator': 'WPS Writer',
 'creationdate': '2026-03-12T11:21:28+05:45',
 'author': 'Dell',
 'comments': '',
 'company': '',
 'keywords': '',
 'moddate': '2026-03-12T11:21:28+05:45',
 'sourcemodified': "D:20260312112128+05'45'",
 'subject': '',
 'title': '',
 'trapped': '/False',
 'source': '../docs/research.pdf',
 'total_pages': 33,
 'page': 0,
 'page_label': '1'}

yedi hamle text dinxau vne afai le banau parxa meta data bcz it accepts '' embedding as data type + metadata ''

other wise for the pdfs it automatically creates metadata.

#### step 3 : Create Embeddings for Chunks

In [20]:
# from langchain_core import OpenAIEmbeddings

# free
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2" 
)

/var/folders/gl/q03cbrwd4398kbj0zgm3bs8h0000gn/T/ipykernel_9307/2677344647.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6682.96it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
embeddings.embed_query("What is the capital of France?")

[0.08204811811447144,
 0.03605550527572632,
 -0.0038928694557398558,
 -0.004881022963672876,
 0.025651102885603905,
 -0.05714346840977669,
 0.012191594578325748,
 0.004678945988416672,
 0.03494986891746521,
 -0.02242191694676876,
 -0.008005285635590553,
 -0.10935352742671967,
 0.022724756971001625,
 -0.029320847243070602,
 -0.04352204501628876,
 -0.1202411875128746,
 -0.0008486209553666413,
 -0.018150117248296738,
 0.056129567325115204,
 0.003085286123678088,
 0.0023363730870187283,
 -0.01683926023542881,
 0.06362468749284744,
 -0.023660236969590187,
 0.03149350360035896,
 -0.03479794040322304,
 -0.020548827946186066,
 -0.0027909704949706793,
 -0.011038015596568584,
 -0.03612672537565231,
 0.05414107069373131,
 -0.03661713749170303,
 -0.025008678436279297,
 -0.03817041590809822,
 -0.04960361868143082,
 -0.015148117206990719,
 0.021315021440386772,
 -0.01274044718593359,
 0.07670088857412338,
 0.04435575753450394,
 -0.010834887623786926,
 -0.029760003089904785,
 -0.01697046123445034,
 -

#### step 4 : Create and Store Embeddings in Vector Store.

In [27]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    
    )

#### step 5 : Semantic Search

In [31]:
vectorstore.similarity_search("what is research topic and which method you are using?", k=4)

[Document(metadata={'total_pages': 33, 'creator': 'WPS Writer', 'comments': '', 'producer': 'macOS Version 26.3.1 (Build 25D2128) Quartz PDFContext', 'source': '../docs/research.pdf', 'company': '', 'title': '', 'trapped': '/False', 'creationdate': '2026-03-12T11:21:28+05:45', 'author': 'Dell', 'subject': '', 'page': 15, 'sourcemodified': "D:20260312112128+05'45'", 'keywords': '', 'page_label': '16', 'moddate': '2026-03-12T11:21:28+05:45'}, page_content='vi. Producing the analytical report, supported by representative participant quotations.\nThe coding process will follow an inductive approach, allowing themes to emerge directly\nfrom the data while remaining aligned with the study’s research objectives.\nThroughout the analysis process, the researchers will maintain reflexive notes to\nacknowledge their perspectives and ensure transparency in interpretation.'),
 Document(metadata={'company': '', 'author': 'Dell', 'producer': 'macOS Version 26.3.1 (Build 25D2128) Quartz PDFContext', '

Talk to LLM

In [32]:
context = vectorstore.similarity_search("RAG",k=3)  

In [34]:
response=llm.invoke(f"what is research topic and which method you are using? You can answer using the following context: {context}")
print(response.content)

**Research Topic:**
The research topic appears to be related to research methodology and project management, specifically in the context of the National Academy of Science and Technology (NAST) and Pokhara University. The topic may involve the development of a research project plan, including systematic planning, monitoring, and achievement of milestones, as well as the preparation of project presentation and viva-voce/defense materials.

**Research Questions:**
Based on the provided context, some potential research questions that could be explored include:

1. What are the key components of a research project plan that ensures systematic planning, monitoring, and achievement of milestones in the context of NAST and Pokhara University?
2. How can project presentation and viva-voce/defense materials be effectively prepared to communicate research findings and recommendations to stakeholders?
3. What are the best practices for maintaining periodic progress reports and ensuring compliance